# Tiền xử lý cho DL / mô hình thống kê (TODO Nhóm 2)

Notebook này thực hiện [TODO.md — Giai đoạn 2.2](../../TODO.md): **log-transform** cho biến dạng nồng độ PM2.5 và **target**, rồi **StandardScaler** (fit **chỉ trên train**) cho toàn bộ ma trận đặc trưng sau khi log.

**Đầu vào:** [`modeling_fs/train_tree.csv`](../data/processed/modeling_fs/train_tree.csv), `val_tree.csv`, `test_tree.csv` (cùng split với notebook 4).

**Đầu ra:** [`train_dl.csv`](../data/processed/modeling_fs/train_dl.csv), `val_dl.csv`, `test_dl.csv` trong **cùng thư mục**; file [`preprocessing_dl.joblib`](../data/processed/preprocessing_dl.joblib) và [`preprocessing_dl_meta.json`](../data/processed/preprocessing_dl_meta.json) để inverse khi đánh giá RMSE/MAE trên thang µg/m³ gốc.

**Lưu ý:** Không scale target (chỉ `log1p`); nếu sau này LSTM cần `y` đã scale, có thể thêm bước riêng.


## Quy tắc biến đổi

1. **Cột áp dụng `log1p`:** `pm25` và mọi cột có tên bắt đầu bằng `pm25_` (lag, rolling, diff, …). Giá trị âm (nếu có do diff) được clip về 0 trước `log1p` để tránh NaN.
2. **Target** `target_pm25_h24` → `target_pm25_h24_log` = `log1p(...)`.
3. **StandardScaler:** `fit` trên **train** sau bước 1, `transform` train/val/test cho **toàn bộ cột đặc trưng** (không gồm `datetime_local` và không gồm target).


In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

BASE = Path("../data/processed")
MODELING = BASE / "modeling_fs"

TARGET_ORIG = "target_pm25_h24"
TARGET_LOG = "target_pm25_h24_log"


In [2]:
def is_pm_family_column(name: str) -> bool:
    return name == "pm25" or name.startswith("pm25_")


def apply_log1p_features(df: pd.DataFrame, cols) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        if is_pm_family_column(c):
            v = out[c].astype(float).values
            v = np.maximum(v, 0.0)
            out[c] = np.log1p(v)
    return out


def apply_log1p_target(s: pd.Series) -> pd.Series:
    v = s.astype(float).values
    v = np.maximum(v, 0.0)
    return pd.Series(np.log1p(v), index=s.index)


In [3]:
train = pd.read_csv(MODELING / "train_tree.csv", parse_dates=["datetime_local"])
val = pd.read_csv(MODELING / "val_tree.csv", parse_dates=["datetime_local"])
test = pd.read_csv(MODELING / "test_tree.csv", parse_dates=["datetime_local"])

feature_cols = [c for c in train.columns if c not in ("datetime_local", TARGET_ORIG)]
print("Số đặc trưng:", len(feature_cols))
log_feature_cols = [c for c in feature_cols if is_pm_family_column(c)]
print("Cột log1p (PM family):", len(log_feature_cols), log_feature_cols[:5], "...")


Số đặc trưng: 26
Cột log1p (PM family): 15 ['pm25', 'pm25_lag_1', 'pm25_roll_72h_mean', 'pm25_roll_6h_mean', 'pm25_lag_24'] ...


In [4]:
train_t = apply_log1p_features(train, feature_cols)
val_t = apply_log1p_features(val, feature_cols)
test_t = apply_log1p_features(test, feature_cols)

train_t[TARGET_LOG] = apply_log1p_target(train[TARGET_ORIG])
val_t[TARGET_LOG] = apply_log1p_target(val[TARGET_ORIG])
test_t[TARGET_LOG] = apply_log1p_target(test[TARGET_ORIG])

X_train = train_t[feature_cols].values.astype(float)
X_val = val_t[feature_cols].values.astype(float)
X_test = test_t[feature_cols].values.astype(float)

scaler = StandardScaler()
scaler.fit(X_train)

X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)
print("Scaler fit xong (train).")


Scaler fit xong (train).


In [5]:
def build_out_df(dt, X_scaled, feat_names, target_log_series):
    out = pd.DataFrame(X_scaled, columns=feat_names)
    # Giữ nguyên chuỗi thời gian (vd +07:00); tránh to_csv tự đổi sang UTC
    out.insert(0, "datetime_local", dt.astype(str))
    out[TARGET_LOG] = target_log_series.values
    return out


train_dl = build_out_df(train["datetime_local"], X_train_s, feature_cols, train_t[TARGET_LOG])
val_dl = build_out_df(val["datetime_local"], X_val_s, feature_cols, val_t[TARGET_LOG])
test_dl = build_out_df(test["datetime_local"], X_test_s, feature_cols, test_t[TARGET_LOG])

train_dl.to_csv(MODELING / "train_dl.csv", index=False)
val_dl.to_csv(MODELING / "val_dl.csv", index=False)
test_dl.to_csv(MODELING / "test_dl.csv", index=False)
print("Đã ghi:", MODELING / "train_dl.csv", val_dl.shape, test_dl.shape)


Đã ghi: ..\data\processed\modeling_fs\train_dl.csv (2128, 28) (2128, 28)


In [6]:
artifact = {
    "scaler": scaler,
    "feature_columns": feature_cols,
    "log1p_feature_columns": log_feature_cols,
    "target_original": TARGET_ORIG,
    "target_transformed": TARGET_LOG,
    "scaler_type": "sklearn.preprocessing.StandardScaler",
}
joblib.dump(artifact, BASE / "preprocessing_dl.joblib")
print("Đã ghi:", BASE / "preprocessing_dl.joblib")


Đã ghi: ..\data\processed\preprocessing_dl.joblib


In [7]:
meta = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_splits": "modeling_fs/train_tree.csv, val_tree.csv, test_tree.csv",
    "target_original": TARGET_ORIG,
    "target_transformed": TARGET_LOG,
    "log1p_feature_columns": log_feature_cols,
    "scaler": "StandardScaler(fit=train only)",
    "inverse_target": "np.expm1(target_pm25_h24_log)",
    "inverse_X_pm_columns": "after scaler.inverse_transform, apply np.expm1 only to columns listed in log1p_feature_columns (by name)",
}
with open(BASE / "preprocessing_dl_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print("Đã ghi:", BASE / "preprocessing_dl_meta.json")


Đã ghi: ..\data\processed\preprocessing_dl_meta.json


## Kiểm tra inverse (minh họa)

Đưa một vài giá trị `target` đã log về µg/m³: `np.expm1`. Đưa `X` về không gian gốc: `inverse_transform` rồi `expm1` chỉ trên các cột PM family.


In [8]:
# Demo: inverse target
orig = train[TARGET_ORIG].iloc[:3].values
logged = train_dl[TARGET_LOG].iloc[:3].values
restored = np.expm1(logged)
print("Gốc:", orig)
print("expm1(log):", restored)
print("Sai số trung bình:", np.abs(orig - restored).max())


Gốc: [44.92916711 46.39999962 54.82083289]
expm1(log): [44.92916711 46.39999962 54.82083289]
Sai số trung bình: 7.105427357601002e-15


In [9]:
# Demo: inverse 1 hàng đặc trưng (vector)
inv_vec = scaler.inverse_transform(X_train_s[:1])
row = pd.Series(inv_vec[0], index=feature_cols)
for c in log_feature_cols:
    row[c] = np.expm1(row[c])
print("Sau inverse scaler + expm1 (chỉ cột PM), pm25:", float(row["pm25"]))
print("So với train gốc pm25:", float(train["pm25"].iloc[0]))


Sau inverse scaler + expm1 (chỉ cột PM), pm25: 20.13333336512248
So với train gốc pm25: 20.13333336512248
